In [1]:
import ccxt
import pandas as pd
import ta
import numpy as np
import vectorbt as vbt
import warnings
warnings.filterwarnings("ignore")
import time

In [ ]:
# check timeframe in each exchange 여러 거래소에서 타임프레임 확인
exchanges = ccxt.upbit()
print(exchanges.timeframes)

{'1s': 'seconds', '1m': 'minutes', '3m': 'minutes', '5m': 'minutes', '10m': 'minutes', '15m': 'minutes', '30m': 'minutes', '1h': 'minutes', '4h': 'minutes', '1d': 'days', '1w': 'weeks', '1M': 'months', '1y': 'years'}


In [2]:
# upbit/bithumb 거래소에서 특정 기간 동안의 XRP/KRW의 1분봉 데이터를 가져오는 함수 정의
def fetch_candles_ohlcv(exchange='upbit', symbol='XRP/KRW', timeframe='1m', start_str='2026-11-01 00:00:00', end_str='2026-11-14 15:59:59'):
    exchange = ccxt.upbit({'enableRateLimit': True}) if exchange=='upbit' else ccxt.bithumb({'enableRateLimit': True})
    
    start_ts = int(pd.Timestamp(start_str, tz='Asia/Seoul').timestamp() * 1000)
    end_ts = int(pd.Timestamp(end_str, tz='Asia/Seoul').timestamp() * 1000)
    
    timeframe_minutes = 1
    timeframe_ms = timeframe_minutes * 60 * 1000
    candle_count = ((end_ts - start_ts) // timeframe_ms) + 1
    
    all_candles = []
    to_ts = end_ts
    
    print(f"[{symbol}] 분봉 데이터 수집 시작 from {start_str} to {end_str}. candle_count: {candle_count/3} ea.")    

    while True:
        params = {'to': exchange.iso8601(to_ts)}
        try:
            candles = exchange.fetch_ohlcv(symbol=symbol, timeframe=timeframe, limit=200, params=params)
        except Exception as e:
            print(f"Error fetching data: {e}. Retrying in 2 seconds...")
            time.sleep(2)
            continue
        
        if not candles:
            break
        
        all_candles = candles + all_candles  # Prepend new candles to the list
        oldest_ts = candles[0][0]  # 첫번째 캔들의 타임스탬프를 가져옵니다.
        
        if oldest_ts <= start_ts:
            break
        
        to_ts = oldest_ts - 1  # Move to the next batch of candles
        time.sleep(0.2)  # Sleep to respect rate limits
        
    seen = set()
    deduped = []
    for c in all_candles:
        if c[0] not in seen and start_ts <= c[0] <= end_ts:
            seen.add(c[0])
            deduped.append(c)
            
    deduped.sort(key=lambda x: x[0])  # Sort by timestamp
    
    df = pd.DataFrame(deduped, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')
    df.set_index('datetime', inplace=True)
    df.drop(columns=['timestamp'], inplace=True)
    return df


In [13]:
df_upb_1xrp = fetch_candles_ohlcv(exchange='upbit', 
                            symbol='BTC/KRW', 
                            timeframe='3m',
                            start_str='2026-11-01 00:00:00',
                            end_str='2026-11-25 23:59:59')

print(f"Fetched {len(df_upb_1xrp)} rows of data for BTC/KRW from Upbit.")
print(df_upb_1xrp.head())
print(df_upb_1xrp.tail())


[BTC/KRW] 분봉 데이터 수집 시작 from 2026-11-01 00:00:00 to 2026-11-25 23:59:59. candle_count: 12000.0 ea.
Fetched 11900 rows of data for BTC/KRW from Upbit.
                                     open            high             low  \
datetime                                                                    
2026-11-01 00:00:00+09:00 104,302,000.000 104,306,000.000 104,302,000.000   
2026-11-01 00:03:00+09:00 104,302,000.000 104,303,000.000 104,168,000.000   
2026-11-01 00:06:00+09:00 104,210,000.000 104,210,000.000 104,101,000.000   
2026-11-01 00:09:00+09:00 104,116,000.000 104,147,000.000 104,093,000.000   
2026-11-01 00:12:00+09:00 104,095,000.000 104,187,000.000 104,091,000.000   

                                    close  volume  
datetime                                           
2026-11-01 00:00:00+09:00 104,302,000.000   1.613  
2026-11-01 00:03:00+09:00 104,211,000.000   3.800  
2026-11-01 00:06:00+09:00 104,116,000.000   1.346  
2026-11-01 00:09:00+09:00 104,095,000.000   5.631  

In [14]:
CONFIG = dict(
    symbol="XRP/KRW",
    exchange="upbit",
    days=30,
    csv_path=None,       # CSV 사용 시 경로 (컬럼: timestamp, open, high, low, close, volume)
    init_cash=100000,
    fees=0.0005,          # 편도 수수료 0.1% (거래소/등급에 맞게 수정)
    slippage=0.0005,     # 편도 슬리피지 0.05% 가정
)
cfg = CONFIG

In [15]:
# [2] ta 라이브러리를 사용하여 이동평균과 RSI 지표를 계산하고, 매수/매도 신호를 생성합니다. by claude

# 추세
df_upb_1xrp['ema09'] = ta.trend.EMAIndicator(close=df_upb_1xrp['close'], window=9).ema_indicator()
df_upb_1xrp['ema21'] = ta.trend.EMAIndicator(close=df_upb_1xrp['close'], window=21).ema_indicator()
macd = ta.trend.MACD(close=df_upb_1xrp['close'], window_slow=13, window_fast=6, window_sign=5)
df_upb_1xrp['macd'], df_upb_1xrp['macd_sig'], df_upb_1xrp['macd_hist'] = macd.macd(), macd.macd_signal(), macd.macd_diff()
df_upb_1xrp['adx'] = ta.trend.ADXIndicator(high=df_upb_1xrp['high'], low=df_upb_1xrp['low'], close=df_upb_1xrp['close'], window=14).adx()

# 모멘텀
df_upb_1xrp['rsi'] = ta.momentum.RSIIndicator(close=df_upb_1xrp['close'], window=7).rsi()
stoch = ta.momentum.StochasticOscillator(high=df_upb_1xrp['high'], low=df_upb_1xrp['low'], close=df_upb_1xrp['close'], window=5, smooth_window=3)
df_upb_1xrp['stoch_k'], df_upb_1xrp['stoch_d'] = stoch.stoch(), stoch.stoch_signal()

# 변동성
bb = ta.volatility.BollingerBands(close=df_upb_1xrp['close'], window=20, window_dev=2)
df_upb_1xrp['bb_up'], df_upb_1xrp['bb_mid'], df_upb_1xrp['bb_low'] = bb.bollinger_hband(), bb.bollinger_mavg(), bb.bollinger_lband()
df_upb_1xrp['bb_width'] = bb.bollinger_wband()
atr = ta.volatility.AverageTrueRange(high=df_upb_1xrp['high'], low=df_upb_1xrp['low'], close=df_upb_1xrp['close'], window=14).average_true_range()
df_upb_1xrp['atr'] = atr.replace(0, np.nan)  # Replace 0 with NaN to avoid division by zero in calculations / ta는 워밍업 구간을 0으로 채우므로 NaN 처리


# 거래량
df_upb_1xrp['vol_ma'] = df_upb_1xrp['volume'].rolling(window=20).mean()
df_upb_1xrp['vol_ratio'] = df_upb_1xrp['volume'] / df_upb_1xrp['vol_ma']

# VWAP: ta의 VWAP는 롤링 윈도우 방식이라, 일(UTC) 단위 앵커드 VWAP를 직접 계산
tp = (df_upb_1xrp['high'] + df_upb_1xrp['low'] + df_upb_1xrp['close']) / 3
day = df_upb_1xrp.index.floor('D')  # 인덱스(연월일 시분초)를 "일" 단위로 내림(floor)하여 그룹화 기준으로 사용(단위는 시간 H, 15분 15m 등도 가능)
df_upb_1xrp['vwap'] = (tp * df_upb_1xrp['volume']).groupby(day).cumsum() / df_upb_1xrp['volume'].groupby(day).cumsum()

# 상위 시간대(5분봉) 추세 필터 — 완성된 5분봉만 사용하도록 shift(1) 후 전방 채움
c5 = df_upb_1xrp['close'].resample("5min").last()
ema5 = ta.trend.EMAIndicator(c5, window=21).ema_indicator()
htf_up = (c5 > ema5).astype(float).shift(1)
df_upb_1xrp["htf_up"] = htf_up.reindex(df_upb_1xrp.index, method="ffill").fillna(0).astype(bool)


In [16]:
# 관통 신호 - a가 b를 상향 돌파하는 시점과 하향 돌파하는 시점을 확인하는 함수 정의
# b가 pd.Series인 경우, b의 이전 값과 비교하여 a가 b를 상향 돌파했는지 확인합니다. pd.Series가 아닌 경우(예, 고정값), b를 그대로 사용하여 비교합니다.
def crossed_above(a, b):
    b_prev = b.shift(1) if isinstance(b, pd.Series) else b
    return (a > b) & (a.shift(1) <= b_prev)

def crossed_below(a, b):
    b_prev = b.shift(1) if isinstance(b, pd.Series) else b
    return (a < b) & (a.shift(1) >= b_prev)

In [17]:
# =============================================================================
# 3. 전략별 매수/매도 신호 반환: entries, exits, sl_stop(비율), tp_stop(비율)
# 전략: (1) 추세추종 눌림목 (2) 평균회귀 (3) 변동성 돌파 
# =============================================================================
def strat_trend_pullback(df, rsi_level=45, vol_mult=1.5, sl_atr=1.0, tp_atr=2.0):
    """가격>VWAP, EMA09>EMA21, 5분봉 상승 중 RSI가 눌림 후 rsi_level 상향 돌파 + 거래량 확인"""
    trend = (df["close"] > df["vwap"]) & (df["ema09"] > df["ema21"]) & df["htf_up"]
    trigger = crossed_above(df["rsi"], rsi_level)
    volume_ok = df["vol_ratio"] > vol_mult
    entries = trend & trigger & volume_ok
    exits = crossed_below(df["ema09"], df["ema21"]) | crossed_below(df["close"], df["vwap"])
    sl = sl_atr * df["atr"] / df["close"]
    tp = tp_atr * df["atr"] / df["close"]
    return entries, exits, sl, tp


def strat_mean_reversion(df, adx_max=20, rsi_max=25, stoch_max=20, sl_atr=1.5):
    """횡보(ADX 낮음) 중 볼린저 하단 이탈 후 복귀 + RSI 과매도 + 스토캐스틱 저점 교차"""
    ranging = df["adx"] < adx_max
    bb_reentry = (df["close"].shift(1) < df["bb_low"].shift(1)) & (df["close"] > df["bb_low"])
    rsi_os = df["rsi"].rolling(3).min() < rsi_max                        # 최근 3봉 내 과매도
    stoch_up = (df["stoch_k"] > df["stoch_d"]) & (df["stoch_k"].rolling(3).min() < stoch_max)
    entries = ranging & bb_reentry & rsi_os & stoch_up
    exits = df["close"] >= df["bb_mid"]                                  # 중심선 도달 시 익절
    sl = sl_atr * df["atr"] / df["close"]
    tp = pd.Series(np.nan, index=df.index)                               # 익절은 exits로 처리
    return entries, exits, sl, tp


def strat_squeeze_breakout(df, squeeze_q=0.1, lookback=120, vol_mult=2.0, sl_atr=0.75, tp_atr=2.0):
    """볼린저 밴드폭이 최근 lookback봉 하위 squeeze_q 분위 → 거래량 동반 상단 돌파"""
    squeeze = df["bb_width"].shift(1) <= df["bb_width"].rolling(lookback).quantile(squeeze_q).shift(1)
    breakout = crossed_above(df["close"], df["bb_up"])
    entries = squeeze & breakout & (df["vol_ratio"] > vol_mult) & df["htf_up"]
    exits = crossed_below(df["close"], df["ema09"])
    sl = sl_atr * df["atr"] / df["close"]
    tp = tp_atr * df["atr"] / df["close"]
    return entries, exits, sl, tp


STRATEGIES = {
    "trend_pullback": strat_trend_pullback,
    "mean_reversion": strat_mean_reversion,
    "squeeze_breakout": strat_squeeze_breakout,
}

In [18]:
# =============================================================================
# 4. 백테스트
# =============================================================================
def run_backtest(df, entries, exits, sl, tp, cfg=CONFIG):
    """
    신호는 봉 마감(close) 기준으로 계산되므로, 룩어헤드를 피하기 위해
    한 봉 뒤로 밀어서 다음 봉 시가(open)에 체결한다.
    손절/익절은 high/low로 봉 내 도달 여부를 판정한다.
    """
    entries = entries.shift(1, fill_value=False).astype(bool)
    exits = exits.shift(1, fill_value=False).astype(bool)
    sl = sl.shift(1)
    tp = tp.shift(1)

    return vbt.Portfolio.from_signals(
        close=df["close"],
        entries=entries,
        exits=exits,
        price=df["open"],              # 다음 봉 시가 체결
        open=df["open"], high=df["high"], low=df["low"],
        sl_stop=sl, tp_stop=tp,        # 진입 시점의 ATR 기반 비율이 적용됨
        stop_entry_price="fillprice",  # 손절/익절 기준 = 실제 체결가
        fees=cfg["fees"],
        slippage=cfg["slippage"],
        init_cash=cfg["init_cash"],
        freq="1min",
    )

In [19]:
def summarize(pf, name):
    trades = pf.trades
    n = trades.count()
    return pd.Series({
        "총수익률(%)": pf.total_return() * 100,
        "Buy&Hold(%)": pf.total_benchmark_return() * 100,
        "최대낙폭(%)": pf.max_drawdown() * 100,
        "샤프": pf.sharpe_ratio(),
        "거래수": n,
        "승률(%)": trades.win_rate() * 100 if n else np.nan,
        "Profit Factor": trades.profit_factor() if n else np.nan,
        "평균보유(분)": trades.duration.mean() if n else np.nan,
        "총수수료": pf.orders.fees.sum(),
    }, name=name)

In [20]:
results, portfolios = [], {}
for name, fn in STRATEGIES.items():
    entries, exits, sl, tp = fn(df_upb_1xrp)
    pf = run_backtest(df_upb_1xrp, entries, exits, sl, tp, cfg)
    portfolios[name] = pf
    results.append(summarize(pf, name))

pd.set_option("display.float_format", "{:,.3f}".format)
print("\n=== 전략 비교 ===")
print(pd.DataFrame(results).T)

best = max(portfolios, key=lambda k: portfolios[k].total_return())
print(f"\n=== 상세 통계: {best} ===")
print(portfolios[best].stats())

# 차트 저장 (브라우저로 열기)
portfolios[best].plot().write_html(f"backtest_{best}.html")
print(f"\n차트 저장: backtest_{best}.html")


=== 전략 비교 ===
               trend_pullback  mean_reversion  squeeze_breakout
총수익률(%)                -2.290          -6.485            -2.704
Buy&Hold(%)           -23.824         -23.824           -23.824
최대낙폭(%)                -2.500          -6.485            -2.704
샤프                    -23.501         -29.162           -27.226
거래수                    14.000          41.000            13.000
승률(%)                  14.286          17.073             0.000
Profit Factor           0.106           0.072             0.000
평균보유(분)                 2.071           3.780             1.231
총수수료                1,382.784       3,957.869         1,284.087

=== 상세 통계: trend_pullback ===
Start                         2026-11-01 00:00:00+09:00
End                           2026-11-25 23:57:00+09:00
Period                                  8 days 06:20:00
Start Value                                 100,000.000
End Value                                    97,710.049
Total Return [%]                  

In [21]:
df_upb_1xrp['entries'], df_upb_1xrp['exits'], df_upb_1xrp['sl'], df_upb_1xrp['tp'] = STRATEGIES[best](df_upb_1xrp)
df_upb_1xrp.head()

,open,high,low,close,volume,ema09,ema21,macd,macd_sig,macd_hist,...,bb_width,atr,vol_ma,vol_ratio,vwap,htf_up,entries,exits,sl,tp
datetime,,,,,,,,,,,,,,,,,,,,,
2026-11-01 00:00:00+09:00,"104,302,000.000","104,306,000.000","104,302,000.000","104,302,000.000",1.613,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"104,303,333.333",False,False,False,NaN,NaN
2026-11-01 00:03:00+09:00,"104,302,000.000","104,303,000.000","104,168,000.000","104,211,000.000",3.800,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"104,249,983.387",False,False,False,NaN,NaN
2026-11-01 00:06:00+09:00,"104,210,000.000","104,210,000.000","104,101,000.000","104,116,000.000",1.346,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"104,228,544.362",False,False,False,NaN,NaN
2026-11-01 00:09:00+09:00,"104,116,000.000","104,147,000.000","104,093,000.000","104,095,000.000",5.631,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"104,175,427.280",False,False,False,NaN,NaN
2026-11-01 00:12:00+09:00,"104,095,000.000","104,187,000.000","104,091,000.000","104,135,000.000",0.766,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"104,173,229.085",False,False,False,NaN,NaN


In [22]:
portfolios["trend_pullback"].plot().show()

In [23]:
portfolios["trend_pullback"].trades.records_readable

,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
0,0,0,0.001,2026-11-01 02:27:00+09:00,"104,541,244.500",49.975,2026-11-01 02:30:00+09:00,"104,467,894.100",49.940,-170.044,-0.002,Long,Closed,0
1,1,0,0.001,2026-11-01 03:45:00+09:00,"104,644,296.000",49.890,2026-11-01 03:48:00+09:00,"104,579,954.796",49.859,-161.100,-0.002,Long,Closed,1
2,2,0,0.001,2026-11-01 05:36:00+09:00,"104,589,268.500",49.810,2026-11-01 05:45:00+09:00,"104,467,000.000",49.751,-216.019,-0.002,Long,Closed,2
3,3,0,0.001,2026-11-02 00:30:00+09:00,"104,691,319.500",49.702,2026-11-02 00:33:00+09:00,"104,628,251.671",49.672,-159.255,-0.002,Long,Closed,3
4,4,0,0.001,2026-11-03 19:33:00+09:00,"103,734,841.500",49.622,2026-11-03 19:36:00+09:00,"103,645,000.000",49.579,-185.153,-0.002,Long,Closed,4
5,5,0,0.001,2026-11-03 22:12:00+09:00,"104,283,115.500",49.529,2026-11-03 22:15:00+09:00,"104,133,000.000",49.458,-241.583,-0.002,Long,Closed,5
6,6,0,0.001,2026-11-04 01:21:00+09:00,"104,364,156.000",49.409,2026-11-04 01:24:00+09:00,"104,169,000.000",49.316,-283.509,-0.003,Long,Closed,6
7,7,0,0.001,2026-11-04 17:03:00+09:00,"105,092,520.000",49.267,2026-11-04 17:15:00+09:00,"105,326,359.218",49.377,120.602,0.001,Long,Closed,7
8,8,0,0.001,2026-11-07 10:21:00+09:00,"103,568,758.500",49.327,2026-11-07 10:27:00+09:00,"103,502,604.430",49.296,-161.638,-0.002,Long,Closed,8
9,9,0,0.001,2026-11-08 13:51:00+09:00,"103,582,765.500",49.247,2026-11-08 13:54:00+09:00,"103,518,215.000",49.216,-159.841,-0.002,Long,Closed,9
